# Session 9.6: A/B Testing & Model Updates

**Duration:** 120 minutes

## Learning Objectives
By the end of this session, you will be able to:
1. Understand model versioning strategies (semantic versioning)
2. Implement A/B testing for machine learning models
3. Compare model performance statistically
4. Execute gradual rollout strategies (canary deployment)
5. Implement rollback procedures for failed deployments
6. Deploy and manage multiple model versions simultaneously
7. Make data-driven decisions about model updates

## Prerequisites
- Completed Sessions 9.1-9.5
- Deployed API with monitoring
- Understanding of hypothesis testing

---
## Part 1: Model Versioning Strategies (15 minutes)

### Why Version Models?

Just like software, ML models need versioning to:
- Track changes and improvements
- Enable rollback if new version fails
- Support A/B testing
- Maintain reproducibility
- Communicate changes to stakeholders

### Semantic Versioning (SemVer)

Format: **MAJOR.MINOR.PATCH** (e.g., v2.1.3)

- **MAJOR** (2): Breaking changes (new features, different input/output)
  - Example: v1.0.0 → v2.0.0 (changed from regression to classification)

- **MINOR** (1): New features, backward compatible
  - Example: v2.0.0 → v2.1.0 (added new features, retrained)

- **PATCH** (3): Bug fixes, small improvements
  - Example: v2.1.0 → v2.1.1 (fixed preprocessing bug)

### When to Update Model Version

| Change | Version Bump | Example |
|--------|--------------|----------|
| New training data | MINOR | v1.2.0 → v1.3.0 |
| New features added | MINOR | v1.2.0 → v1.3.0 |
| Hyperparameter tuning | PATCH | v1.2.0 → v1.2.1 |
| Bug fix in preprocessing | PATCH | v1.2.0 → v1.2.1 |
| Changed algorithm | MAJOR | v1.2.0 → v2.0.0 |
| Changed target variable | MAJOR | v1.2.0 → v2.0.0 |
| API breaking change | MAJOR | v1.2.0 → v2.0.0 |

### Model Metadata

In [ ]:
import json
from datetime import datetime
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

class ModelVersion:
    """Manage model versions with metadata."""
    
    def __init__(self, version, model, metadata=None):
        self.version = version
        self.model = model
        self.metadata = metadata or {}
        self.metadata['version'] = version
        self.metadata['created_at'] = datetime.now().isoformat()
    
    def save(self, path_prefix='model'):
        """Save model and metadata."""
        model_path = f"{path_prefix}_{self.version}.pkl"
        metadata_path = f"{path_prefix}_{self.version}_metadata.json"
        
        # Save model
        joblib.dump(self.model, model_path)
        
        # Save metadata
        with open(metadata_path, 'w') as f:
            json.dump(self.metadata, f, indent=2)
        
        print(f"✓ Saved model {self.version}")
        print(f"  Model: {model_path}")
        print(f"  Metadata: {metadata_path}")
        return model_path, metadata_path
    
    @classmethod
    def load(cls, version, path_prefix='model'):
        """Load model and metadata."""
        model_path = f"{path_prefix}_{version}.pkl"
        metadata_path = f"{path_prefix}_{version}_metadata.json"
        
        # Load model
        model = joblib.load(model_path)
        
        # Load metadata
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
        
        return cls(version, model, metadata)

# Example: Create versioned models
print("Creating model metadata example...\n")

# Generate sample data
np.random.seed(42)
X = np.random.randn(1000, 5)
y = X[:, 0] * 2 + X[:, 1] * -1 + np.random.randn(1000) * 0.5

# Train v1.0.0
model_v1 = Ridge(alpha=1.0)
model_v1.fit(X, y)

version_v1 = ModelVersion(
    version='v1.0.0',
    model=model_v1,
    metadata={
        'algorithm': 'Ridge',
        'hyperparameters': {'alpha': 1.0},
        'training_samples': len(X),
        'features': 5,
        'train_r2': r2_score(y, model_v1.predict(X)),
        'description': 'Initial baseline model'
    }
)

print("Model v1.0.0 Metadata:")
print(json.dumps(version_v1.metadata, indent=2))

### Deployment Strategies

#### 1. Blue-Green Deployment

- **Blue**: Current production model
- **Green**: New model version
- Switch traffic instantly from blue to green
- Easy rollback (switch back to blue)

#### 2. Canary Deployment

- Deploy new version to small % of traffic (5-10%)
- Monitor performance
- Gradually increase traffic if successful
- Rollback if issues detected

#### 3. A/B Testing

- Run both versions simultaneously
- Split traffic (e.g., 50/50)
- Compare performance statistically
- Choose winner based on data

---
## Part 2: A/B Testing for ML (25 minutes)

### What is A/B Testing?

A/B testing (split testing) compares two versions to determine which performs better.

**Process:**
1. Define metric (accuracy, precision, user engagement)
2. Split traffic between versions
3. Collect data
4. Analyze results statistically
5. Make decision

### Metrics to Compare

**Model Metrics:**
- Accuracy, precision, recall, F1
- RMSE, MAE, R²
- Confidence scores
- Prediction distribution

**User Metrics:**
- Click-through rate
- Conversion rate
- User satisfaction
- Engagement time

**System Metrics:**
- Response time
- Error rate
- Resource usage

### Traffic Splitting Strategies

In [ ]:
import random
from collections import defaultdict

class ABTestRouter:
    """Route traffic between model versions for A/B testing."""
    
    def __init__(self, versions, weights=None):
        """
        Initialize A/B test router.
        
        Args:
            versions: Dictionary of {version_name: model}
            weights: Dictionary of {version_name: weight} (must sum to 1.0)
        """
        self.versions = versions
        self.weights = weights or {v: 1.0/len(versions) for v in versions}
        self.results = defaultdict(list)
        
        # Validate weights
        if abs(sum(self.weights.values()) - 1.0) > 1e-6:
            raise ValueError("Weights must sum to 1.0")
    
    def assign_version(self, user_id=None):
        """
        Assign a version for prediction.
        
        Args:
            user_id: Optional user ID for consistent assignment
        
        Returns:
            version_name: Assigned version
        """
        if user_id is not None:
            # Consistent hashing: same user always gets same version
            random.seed(hash(user_id))
        
        # Weighted random choice
        rand = random.random()
        cumsum = 0
        for version, weight in self.weights.items():
            cumsum += weight
            if rand <= cumsum:
                return version
        
        return list(self.versions.keys())[-1]  # Fallback
    
    def predict(self, X, user_id=None):
        """
        Make prediction using assigned version.
        
        Args:
            X: Input features
            user_id: Optional user ID
        
        Returns:
            (prediction, version_used)
        """
        version = self.assign_version(user_id)
        model = self.versions[version]
        prediction = model.predict([X])[0]
        
        # Record result
        self.results[version].append(prediction)
        
        return prediction, version
    
    def get_statistics(self):
        """Get statistics for each version."""
        stats = {}
        for version, predictions in self.results.items():
            if predictions:
                stats[version] = {
                    'count': len(predictions),
                    'mean': np.mean(predictions),
                    'std': np.std(predictions),
                    'min': np.min(predictions),
                    'max': np.max(predictions)
                }
        return stats

# Example: 50/50 split
print("Example 1: 50/50 A/B Test\n")

# Train two models
model_a = Ridge(alpha=1.0)
model_b = Ridge(alpha=0.1)  # Different hyperparameter
model_a.fit(X, y)
model_b.fit(X, y)

# Create A/B router
router = ABTestRouter(
    versions={'v1.0.0': model_a, 'v1.1.0': model_b},
    weights={'v1.0.0': 0.5, 'v1.1.0': 0.5}
)

# Simulate predictions
version_counts = defaultdict(int)
for i in range(1000):
    x_test = np.random.randn(5)
    pred, version = router.predict(x_test)
    version_counts[version] += 1

print("Traffic distribution:")
for version, count in version_counts.items():
    print(f"  {version}: {count} requests ({count/1000*100:.1f}%)")

print("\nPrediction statistics:")
stats = router.get_statistics()
for version, s in stats.items():
    print(f"\n{version}:")
    print(f"  Count: {s['count']}")
    print(f"  Mean: {s['mean']:.3f}")
    print(f"  Std: {s['std']:.3f}")

### Statistical Significance Testing

To determine if performance difference is real (not due to chance):

In [ ]:
from scipy import stats

def compare_models_statistically(predictions_a, predictions_b, ground_truth_a=None, ground_truth_b=None):
    """
    Compare two models statistically.
    
    Args:
        predictions_a: Predictions from model A
        predictions_b: Predictions from model B
        ground_truth_a: Actual values for A (optional)
        ground_truth_b: Actual values for B (optional)
    
    Returns:
        Dictionary with comparison results
    """
    results = {}
    
    # Compare prediction distributions
    t_stat, p_value = stats.ttest_ind(predictions_a, predictions_b)
    
    # Effect size (Cohen's d)
    mean_diff = np.mean(predictions_b) - np.mean(predictions_a)
    pooled_std = np.sqrt((np.var(predictions_a) + np.var(predictions_b)) / 2)
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    
    results['t_test'] = {
        'statistic': float(t_stat),
        'p_value': float(p_value),
        'significant': p_value < 0.05,
        'interpretation': 'Significant difference' if p_value < 0.05 else 'No significant difference'
    }
    
    results['effect_size'] = {
        'cohens_d': float(cohens_d),
        'interpretation': interpret_cohens_d(cohens_d)
    }
    
    results['descriptive'] = {
        'model_a': {
            'n': len(predictions_a),
            'mean': float(np.mean(predictions_a)),
            'std': float(np.std(predictions_a))
        },
        'model_b': {
            'n': len(predictions_b),
            'mean': float(np.mean(predictions_b)),
            'std': float(np.std(predictions_b))
        },
        'difference': {
            'absolute': float(mean_diff),
            'relative_pct': float(mean_diff / np.mean(predictions_a) * 100) if np.mean(predictions_a) != 0 else 0
        }
    }
    
    # If ground truth available, compare errors
    if ground_truth_a is not None and ground_truth_b is not None:
        errors_a = np.abs(np.array(predictions_a) - np.array(ground_truth_a))
        errors_b = np.abs(np.array(predictions_b) - np.array(ground_truth_b))
        
        # Compare error distributions
        t_stat_err, p_value_err = stats.ttest_ind(errors_a, errors_b)
        
        results['error_comparison'] = {
            'model_a_mae': float(np.mean(errors_a)),
            'model_b_mae': float(np.mean(errors_b)),
            't_statistic': float(t_stat_err),
            'p_value': float(p_value_err),
            'significant': p_value_err < 0.05,
            'winner': 'Model B' if np.mean(errors_b) < np.mean(errors_a) else 'Model A'
        }
    
    return results

def interpret_cohens_d(d):
    """Interpret Cohen's d effect size."""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"

# Example comparison
print("\nExample 2: Statistical Comparison\n")

# Get predictions from both models
preds_a = router.results['v1.0.0']
preds_b = router.results['v1.1.0']

comparison = compare_models_statistically(preds_a, preds_b)

print("Statistical Comparison Results:")
print(f"\nT-Test: {comparison['t_test']['interpretation']}")
print(f"  p-value: {comparison['t_test']['p_value']:.4f}")
print(f"\nEffect Size: {comparison['effect_size']['interpretation']}")
print(f"  Cohen's d: {comparison['effect_size']['cohens_d']:.3f}")
print(f"\nModel A: mean={comparison['descriptive']['model_a']['mean']:.3f}")
print(f"Model B: mean={comparison['descriptive']['model_b']['mean']:.3f}")
print(f"Difference: {comparison['descriptive']['difference']['relative_pct']:.2f}%")

### Sample Size and Test Duration

**How long to run the test?**

- Need enough samples for statistical power
- Typical: 1000-10000 predictions per version minimum
- Duration depends on traffic volume

**Sample Size Calculator:**

In [ ]:
def calculate_required_sample_size(baseline_mean, min_detectable_effect, std, alpha=0.05, power=0.8):
    """
    Calculate required sample size for A/B test.
    
    Args:
        baseline_mean: Current model's mean metric
        min_detectable_effect: Minimum % improvement to detect (e.g., 0.05 for 5%)
        std: Standard deviation of metric
        alpha: Significance level (typically 0.05)
        power: Statistical power (typically 0.8)
    
    Returns:
        Required sample size per variant
    """
    from scipy.stats import norm
    
    # Effect size
    effect_size = (baseline_mean * min_detectable_effect) / std
    
    # Z-scores
    z_alpha = norm.ppf(1 - alpha/2)
    z_beta = norm.ppf(power)
    
    # Sample size formula
    n = 2 * ((z_alpha + z_beta) / effect_size) ** 2
    
    return int(np.ceil(n))

# Example
baseline_mean = 100
std = 10
min_effect = 0.05  # Want to detect 5% improvement

required_n = calculate_required_sample_size(baseline_mean, min_effect, std)
print(f"\nRequired sample size per model: {required_n:,}")
print(f"Total predictions needed: {required_n * 2:,}")
print(f"\nIf you get 1000 predictions/day:")
print(f"  Test duration: {required_n * 2 / 1000:.1f} days")

---
## Part 3: Gradual Rollout (20 minutes)

### Canary Deployment Strategy

Instead of switching all traffic at once, gradually increase:

```
Stage 1: 5% new model, 95% old model
  ↓ (Monitor for 24h)
Stage 2: 25% new model, 75% old model
  ↓ (Monitor for 24h)
Stage 3: 50% new model, 50% old model
  ↓ (Monitor for 48h)
Stage 4: 100% new model
```

### Implementing Gradual Rollout

In [ ]:
class CanaryDeployment:
    """Manage gradual rollout of new model version."""
    
    def __init__(self, old_model, new_model, initial_weight=0.05):
        """
        Initialize canary deployment.
        
        Args:
            old_model: Current production model
            new_model: New model to deploy
            initial_weight: Initial traffic % for new model (default 5%)
        """
        self.old_model = old_model
        self.new_model = new_model
        self.new_weight = initial_weight
        self.results = {'old': [], 'new': []}
        self.stage = 1
    
    def predict(self, X):
        """Make prediction using weighted routing."""
        if random.random() < self.new_weight:
            pred = self.new_model.predict([X])[0]
            self.results['new'].append(pred)
            return pred, 'new'
        else:
            pred = self.old_model.predict([X])[0]
            self.results['old'].append(pred)
            return pred, 'old'
    
    def get_metrics(self):
        """Get current metrics for both models."""
        metrics = {}
        for version in ['old', 'new']:
            if self.results[version]:
                metrics[version] = {
                    'count': len(self.results[version]),
                    'mean': np.mean(self.results[version]),
                    'std': np.std(self.results[version])
                }
        return metrics
    
    def should_increase_traffic(self, threshold=0.05, min_samples=100):
        """
        Decide if it's safe to increase traffic to new model.
        
        Args:
            threshold: P-value threshold for significance
            min_samples: Minimum samples before increasing
        
        Returns:
            (decision, reason)
        """
        # Need minimum samples
        if len(self.results['new']) < min_samples:
            return False, f"Need {min_samples - len(self.results['new'])} more samples"
        
        # Compare distributions
        if len(self.results['old']) > 0 and len(self.results['new']) > 0:
            t_stat, p_value = stats.ttest_ind(self.results['old'], self.results['new'])
            
            # New model significantly worse?
            if p_value < threshold and np.mean(self.results['new']) < np.mean(self.results['old']):
                return False, f"New model performing worse (p={p_value:.4f})"
            
            # Safe to proceed
            return True, f"Metrics look good (p={p_value:.4f})"
        
        return False, "Insufficient data"
    
    def increase_traffic(self, increment=0.20):
        """Increase traffic to new model."""
        old_weight = self.new_weight
        self.new_weight = min(1.0, self.new_weight + increment)
        self.stage += 1
        print(f"\n📈 Stage {self.stage}: Increased new model traffic")
        print(f"   {old_weight*100:.0f}% → {self.new_weight*100:.0f}%")
    
    def rollback(self):
        """Rollback to old model."""
        self.new_weight = 0.0
        print("\n🔄 ROLLBACK: Reverted to old model (0% new traffic)")
    
    def complete_rollout(self):
        """Complete rollout to 100% new model."""
        self.new_weight = 1.0
        print("\n✅ ROLLOUT COMPLETE: 100% traffic on new model")

# Example: Gradual rollout simulation
print("\nExample 3: Canary Deployment Simulation\n")
print("="*60)

# Create canary deployment
canary = CanaryDeployment(
    old_model=model_a,
    new_model=model_b,
    initial_weight=0.05  # Start with 5%
)

print(f"Stage 1: Starting with {canary.new_weight*100:.0f}% new model traffic")

# Simulate traffic
stages = [
    (200, 0.05, "Initial canary"),
    (200, 0.25, "Increased to 25%"),
    (300, 0.50, "Increased to 50%"),
    (300, 1.00, "Full rollout")
]

for stage_requests, target_weight, stage_name in stages:
    # Generate predictions
    for _ in range(stage_requests):
        x_test = np.random.randn(5)
        pred, version = canary.predict(x_test)
    
    # Check metrics
    metrics = canary.get_metrics()
    print(f"\n{stage_name}:")
    print(f"  Old model: {metrics.get('old', {}).get('count', 0)} requests")
    print(f"  New model: {metrics.get('new', {}).get('count', 0)} requests")
    
    # Decide whether to proceed
    if canary.new_weight < target_weight:
        proceed, reason = canary.should_increase_traffic(min_samples=50)
        print(f"  Decision: {reason}")
        
        if proceed:
            canary.increase_traffic(increment=target_weight - canary.new_weight)
        else:
            print("  ⚠️ Holding at current traffic level")
            break

print("\n" + "="*60)
final_metrics = canary.get_metrics()
print("\nFinal Statistics:")
for version in ['old', 'new']:
    if version in final_metrics:
        m = final_metrics[version]
        print(f"\n{version.upper()} model:")
        print(f"  Requests: {m['count']}")
        print(f"  Mean: {m['mean']:.3f}")
        print(f"  Std: {m['std']:.3f}")

### Monitoring During Rollout

In [ ]:
# Visualize rollout progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Traffic distribution
old_count = final_metrics.get('old', {}).get('count', 0)
new_count = final_metrics.get('new', {}).get('count', 0)

ax1.pie(
    [old_count, new_count],
    labels=['Old Model', 'New Model'],
    autopct='%1.1f%%',
    colors=['#ff7f0e', '#2ca02c'],
    startangle=90
)
ax1.set_title('Traffic Distribution', fontsize=14, fontweight='bold')

# Prediction comparison
if 'old' in final_metrics and 'new' in final_metrics:
    ax2.hist(canary.results['old'], bins=30, alpha=0.6, label='Old Model', edgecolor='black')
    ax2.hist(canary.results['new'], bins=30, alpha=0.6, label='New Model', edgecolor='black')
    ax2.axvline(final_metrics['old']['mean'], color='orange', linestyle='--', linewidth=2,
                label=f"Old Mean: {final_metrics['old']['mean']:.2f}")
    ax2.axvline(final_metrics['new']['mean'], color='green', linestyle='--', linewidth=2,
                label=f"New Mean: {final_metrics['new']['mean']:.2f}")
    ax2.set_xlabel('Prediction Value', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('Prediction Distribution Comparison', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Part 4: Rollback Procedures (15 minutes)

### When to Rollback

Immediately rollback if you detect:

1. **Performance degradation**: New model significantly worse
2. **Error rate spike**: More errors/crashes
3. **Response time increase**: Unacceptable latency
4. **User complaints**: Negative feedback
5. **Data quality issues**: Prediction anomalies

### Automated Rollback

In [ ]:
class AutomatedRollback:
    """Automatic rollback based on metrics."""
    
    def __init__(self, canary_deployment, rollback_thresholds=None):
        self.canary = canary_deployment
        self.thresholds = rollback_thresholds or {
            'performance_drop': 0.10,  # 10% worse performance
            'min_samples': 50,
            'p_value': 0.05
        }
        self.rollback_triggered = False
    
    def check_rollback_conditions(self):
        """
        Check if rollback is needed.
        
        Returns:
            (should_rollback, reasons)
        """
        reasons = []
        
        # Need minimum samples
        if len(self.canary.results['new']) < self.thresholds['min_samples']:
            return False, ["Insufficient data for rollback decision"]
        
        if len(self.canary.results['old']) == 0:
            return False, ["No baseline data"]
        
        # Get metrics
        old_mean = np.mean(self.canary.results['old'])
        new_mean = np.mean(self.canary.results['new'])
        
        # Check 1: Performance drop
        performance_drop = (old_mean - new_mean) / old_mean
        if performance_drop > self.thresholds['performance_drop']:
            reasons.append(
                f"Performance dropped {performance_drop*100:.1f}% "
                f"(threshold: {self.thresholds['performance_drop']*100:.1f}%)"
            )
        
        # Check 2: Statistical significance of degradation
        t_stat, p_value = stats.ttest_ind(self.canary.results['old'], self.canary.results['new'])
        if p_value < self.thresholds['p_value'] and new_mean < old_mean:
            reasons.append(
                f"Statistically significant degradation detected (p={p_value:.4f})"
            )
        
        # Check 3: Anomalous predictions
        new_std = np.std(self.canary.results['new'])
        old_std = np.std(self.canary.results['old'])
        if new_std > old_std * 2:
            reasons.append(
                f"High variance detected in new model (std: {new_std:.2f} vs {old_std:.2f})"
            )
        
        should_rollback = len(reasons) > 0
        return should_rollback, reasons
    
    def execute_if_needed(self):
        """Check conditions and execute rollback if needed."""
        should_rollback, reasons = self.check_rollback_conditions()
        
        if should_rollback and not self.rollback_triggered:
            print("\n🚨 AUTOMATED ROLLBACK TRIGGERED")
            print("\nReasons:")
            for reason in reasons:
                print(f"  ⚠️ {reason}")
            
            self.canary.rollback()
            self.rollback_triggered = True
            
            # Log rollback event
            self._log_rollback_event(reasons)
            
            return True
        
        return False
    
    def _log_rollback_event(self, reasons):
        """Log rollback event for post-mortem."""
        event = {
            'timestamp': datetime.now().isoformat(),
            'type': 'AUTOMATED_ROLLBACK',
            'reasons': reasons,
            'metrics': self.canary.get_metrics()
        }
        
        # In production: save to database or monitoring system
        print("\n📝 Rollback event logged:")
        print(json.dumps(event, indent=2))

# Example: Simulate rollback scenario
print("\nExample 4: Automated Rollback\n")
print("="*60)

# Create a worse model for rollback scenario
model_worse = Ridge(alpha=10.0)  # Much higher regularization = worse fit
model_worse.fit(X, y + np.random.randn(len(y)) * 5)  # Add noise

canary_rollback = CanaryDeployment(
    old_model=model_a,
    new_model=model_worse,
    initial_weight=0.20
)

auto_rollback = AutomatedRollback(canary_rollback)

print("Deploying potentially problematic model...\n")

# Simulate traffic
for i in range(200):
    x_test = np.random.randn(5)
    pred, version = canary_rollback.predict(x_test)
    
    # Check rollback conditions periodically
    if i > 0 and i % 50 == 0:
        print(f"\nAfter {i} requests:")
        metrics = canary_rollback.get_metrics()
        print(f"  Old model mean: {metrics.get('old', {}).get('mean', 0):.3f}")
        print(f"  New model mean: {metrics.get('new', {}).get('mean', 0):.3f}")
        
        if auto_rollback.execute_if_needed():
            break

print("\n" + "="*60)

### Post-Mortem Analysis

After a rollback, analyze what went wrong:

1. **What triggered the rollback?**
   - Performance metrics
   - Error rates
   - User feedback

2. **Root cause analysis**
   - Training data issues?
   - Hyperparameter problems?
   - Code bugs?
   - Infrastructure issues?

3. **Prevention**
   - Better testing
   - Improved validation
   - Enhanced monitoring

4. **Documentation**
   - Record incident
   - Share learnings
   - Update procedures

---
## Part 5: Complete A/B Testing Example (40 minutes)

Let's put it all together with a complete stock predictor A/B test!

### Scenario

We have:
- **Model v1.0.0**: Random Forest (current production)
- **Model v1.1.0**: Improved Random Forest (more trees, better features)

Goal: Determine which model performs better in production.

In [ ]:
# Step 1: Generate realistic stock data
print("Step 1: Generating stock price data...\n")

np.random.seed(42)
n_samples = 2000

# Create features
data = pd.DataFrame({
    'day_of_year': np.random.randint(1, 366, n_samples),
    'day_of_week': np.random.randint(0, 7, n_samples),
    'trend': np.arange(n_samples),
    'price_lag_1': 100 + np.cumsum(np.random.randn(n_samples) * 2),
    'price_lag_7': 100 + np.cumsum(np.random.randn(n_samples) * 1.5),
    'volume': np.random.randint(1000000, 5000000, n_samples)
})

# Target: next day price
data['price'] = (
    data['price_lag_1'] * 0.7 +
    data['price_lag_7'] * 0.2 +
    np.sin(2 * np.pi * data['day_of_year'] / 365) * 5 +
    np.random.randn(n_samples) * 3
)

# Split into train and test
train_size = 1500
X_train = data[['day_of_year', 'day_of_week', 'trend', 'price_lag_1', 'price_lag_7', 'volume']][:train_size]
y_train = data['price'][:train_size]
X_test = data[['day_of_year', 'day_of_week', 'trend', 'price_lag_1', 'price_lag_7', 'volume']][train_size:]
y_test = data['price'][train_size:]

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

# Step 2: Train both models
print("\nStep 2: Training models...\n")

# Model v1.0.0 (baseline)
model_v1 = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
model_v1.fit(X_train, y_train)
y_pred_v1 = model_v1.predict(X_test)
rmse_v1 = np.sqrt(mean_squared_error(y_test, y_pred_v1))
r2_v1 = r2_score(y_test, y_pred_v1)

print(f"Model v1.0.0 (baseline):")
print(f"  RMSE: {rmse_v1:.3f}")
print(f"  R²: {r2_v1:.3f}")

# Model v1.1.0 (improved)
model_v2 = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42)
model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_test)
rmse_v2 = np.sqrt(mean_squared_error(y_test, y_pred_v2))
r2_v2 = r2_score(y_test, y_pred_v2)

print(f"\nModel v1.1.0 (improved):")
print(f"  RMSE: {rmse_v2:.3f}")
print(f"  R²: {r2_v2:.3f}")
print(f"\nImprovement: {(rmse_v1 - rmse_v2) / rmse_v1 * 100:.1f}% better RMSE")

In [ ]:
# Step 3: Run A/B test
print("\nStep 3: Running A/B test (50/50 split)...\n")

ab_router = ABTestRouter(
    versions={'v1.0.0': model_v1, 'v1.1.0': model_v2},
    weights={'v1.0.0': 0.5, 'v1.1.0': 0.5}
)

# Track predictions and ground truth
ab_results = {'v1.0.0': {'predictions': [], 'truth': []},
              'v1.1.0': {'predictions': [], 'truth': []}}

# Simulate production traffic
for i in range(len(X_test)):
    x = X_test.iloc[i].values
    actual = y_test.iloc[i]
    
    pred, version = ab_router.predict(x)
    
    ab_results[version]['predictions'].append(pred)
    ab_results[version]['truth'].append(actual)

print("A/B Test Complete!")
print(f"\nTraffic distribution:")
for version in ['v1.0.0', 'v1.1.0']:
    count = len(ab_results[version]['predictions'])
    print(f"  {version}: {count} requests ({count/len(X_test)*100:.1f}%)")

In [ ]:
# Step 4: Analyze results
print("\nStep 4: Analyzing A/B test results...\n")
print("="*60)

results_comparison = {}

for version in ['v1.0.0', 'v1.1.0']:
    preds = np.array(ab_results[version]['predictions'])
    truth = np.array(ab_results[version]['truth'])
    
    errors = np.abs(preds - truth)
    
    results_comparison[version] = {
        'n': len(preds),
        'mae': np.mean(errors),
        'rmse': np.sqrt(np.mean(errors**2)),
        'r2': r2_score(truth, preds),
        'mean_prediction': np.mean(preds),
        'std_prediction': np.std(preds)
    }

print("\nPERFORMANCE COMPARISON")
print("="*60)

for version in ['v1.0.0', 'v1.1.0']:
    r = results_comparison[version]
    print(f"\n{version}:")
    print(f"  Samples: {r['n']}")
    print(f"  MAE: {r['mae']:.3f}")
    print(f"  RMSE: {r['rmse']:.3f}")
    print(f"  R²: {r['r2']:.3f}")

# Calculate improvement
mae_improvement = (results_comparison['v1.0.0']['mae'] - results_comparison['v1.1.0']['mae']) / results_comparison['v1.0.0']['mae']
rmse_improvement = (results_comparison['v1.0.0']['rmse'] - results_comparison['v1.1.0']['rmse']) / results_comparison['v1.0.0']['rmse']

print(f"\nIMPROVEMENT:")
print(f"  MAE: {mae_improvement*100:+.1f}%")
print(f"  RMSE: {rmse_improvement*100:+.1f}%")

In [ ]:
# Step 5: Statistical significance
print("\nStep 5: Statistical Significance Testing...\n")
print("="*60)

errors_v1 = np.abs(np.array(ab_results['v1.0.0']['predictions']) - np.array(ab_results['v1.0.0']['truth']))
errors_v2 = np.abs(np.array(ab_results['v1.1.0']['predictions']) - np.array(ab_results['v1.1.0']['truth']))

# T-test on errors (lower is better)
t_stat, p_value = stats.ttest_ind(errors_v1, errors_v2)

# Effect size
mean_diff = errors_v2.mean() - errors_v1.mean()
pooled_std = np.sqrt((errors_v1.var() + errors_v2.var()) / 2)
cohens_d = mean_diff / pooled_std

print("\nSTATISTICAL TESTS:")
print(f"  T-statistic: {t_stat:.4f}")
print(f"  P-value: {p_value:.4f}")
print(f"  Significant: {'YES' if p_value < 0.05 else 'NO'} (α=0.05)")
print(f"\n  Cohen's d: {cohens_d:.3f}")
print(f"  Effect size: {interpret_cohens_d(cohens_d)}")

# Decision
print("\n" + "="*60)
print("DECISION:")
print("="*60)

if p_value < 0.05 and errors_v2.mean() < errors_v1.mean():
    print("\n✅ DEPLOY v1.1.0")
    print("\nReasons:")
    print(f"  • Statistically significant improvement (p={p_value:.4f})")
    print(f"  • {abs(cohens_d):.1f}x effect size ({interpret_cohens_d(cohens_d)})")
    print(f"  • {abs(mae_improvement)*100:.1f}% better MAE")
    print(f"  • {abs(rmse_improvement)*100:.1f}% better RMSE")
elif p_value < 0.05 and errors_v2.mean() > errors_v1.mean():
    print("\n❌ KEEP v1.0.0 (v1.1.0 is WORSE)")
    print("\nReasons:")
    print(f"  • New model performs significantly worse (p={p_value:.4f})")
    print("  • Roll back and investigate issues")
else:
    print("\n⚖️ NO SIGNIFICANT DIFFERENCE")
    print("\nReasons:")
    print(f"  • No statistically significant difference (p={p_value:.4f})")
    print("  • Consider keeping v1.0.0 (simpler is better)")
    print("  • Or run longer test for more data")

In [ ]:
# Step 6: Visualization
print("\nStep 6: Visualizing results...\n")

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 2, hspace=0.3, wspace=0.3)

# 1. Error distribution comparison
ax1 = fig.add_subplot(gs[0, :])
ax1.hist(errors_v1, bins=30, alpha=0.6, label='v1.0.0', edgecolor='black')
ax1.hist(errors_v2, bins=30, alpha=0.6, label='v1.1.0', edgecolor='black')
ax1.axvline(errors_v1.mean(), color='blue', linestyle='--', linewidth=2,
            label=f'v1.0.0 Mean: {errors_v1.mean():.2f}')
ax1.axvline(errors_v2.mean(), color='orange', linestyle='--', linewidth=2,
            label=f'v1.1.0 Mean: {errors_v2.mean():.2f}')
ax1.set_xlabel('Absolute Error', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title(f'Error Distribution (p-value: {p_value:.4f})', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Prediction accuracy scatter (v1.0.0)
ax2 = fig.add_subplot(gs[1, 0])
ax2.scatter(ab_results['v1.0.0']['truth'], ab_results['v1.0.0']['predictions'],
            alpha=0.5, s=20)
ax2.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)],
         'r--', linewidth=2, label='Perfect prediction')
ax2.set_xlabel('Actual Price', fontsize=11)
ax2.set_ylabel('Predicted Price', fontsize=11)
ax2.set_title(f'v1.0.0: R²={results_comparison["v1.0.0"]["r2"]:.3f}',
              fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Prediction accuracy scatter (v1.1.0)
ax3 = fig.add_subplot(gs[1, 1])
ax3.scatter(ab_results['v1.1.0']['truth'], ab_results['v1.1.0']['predictions'],
            alpha=0.5, s=20, color='orange')
ax3.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)],
         'r--', linewidth=2, label='Perfect prediction')
ax3.set_xlabel('Actual Price', fontsize=11)
ax3.set_ylabel('Predicted Price', fontsize=11)
ax3.set_title(f'v1.1.0: R²={results_comparison["v1.1.0"]["r2"]:.3f}',
              fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. Metrics comparison
ax4 = fig.add_subplot(gs[2, 0])
metrics = ['MAE', 'RMSE']
v1_metrics = [results_comparison['v1.0.0']['mae'], results_comparison['v1.0.0']['rmse']]
v2_metrics = [results_comparison['v1.1.0']['mae'], results_comparison['v1.1.0']['rmse']]

x = np.arange(len(metrics))
width = 0.35
ax4.bar(x - width/2, v1_metrics, width, label='v1.0.0', alpha=0.8)
ax4.bar(x + width/2, v2_metrics, width, label='v1.1.0', alpha=0.8)
ax4.set_ylabel('Error', fontsize=11)
ax4.set_title('Performance Metrics Comparison', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.grid(True, alpha=0.3, axis='y')

# 5. Sample size and power
ax5 = fig.add_subplot(gs[2, 1])
categories = ['v1.0.0', 'v1.1.0']
counts = [results_comparison['v1.0.0']['n'], results_comparison['v1.1.0']['n']]
colors_pie = ['#1f77b4', '#ff7f0e']
ax5.pie(counts, labels=categories, autopct='%1.1f%%', colors=colors_pie, startangle=90)
ax5.set_title(f'Traffic Distribution (Total: {sum(counts)})', fontsize=12, fontweight='bold')

plt.suptitle('A/B Test Results: Model v1.0.0 vs v1.1.0', fontsize=16, fontweight='bold', y=0.995)
plt.show()

---
## Summary & Best Practices

### What You Learned

✅ Model versioning strategies (semantic versioning)
✅ A/B testing implementation for ML models
✅ Statistical comparison of model performance
✅ Gradual rollout (canary deployment)
✅ Automated rollback procedures
✅ Making data-driven deployment decisions

### Deployment Decision Framework

```
1. Offline Evaluation
   ↓ (Passes quality bar?)
2. Canary Deployment (5% traffic)
   ↓ (No issues after 24h?)
3. A/B Test (50% traffic)
   ↓ (Statistically significant improvement?)
4. Gradual Rollout (50% → 100%)
   ↓ (Continuous monitoring)
5. Full Deployment
```

### Best Practices Checklist

**Before Deployment:**
- [ ] Thorough offline evaluation
- [ ] Version model with metadata
- [ ] Document changes and improvements
- [ ] Define rollback criteria
- [ ] Set up monitoring and alerts

**During A/B Test:**
- [ ] Ensure random traffic assignment
- [ ] Collect sufficient sample size
- [ ] Monitor both statistical and practical significance
- [ ] Track system metrics (latency, errors)
- [ ] Be ready to rollback

**After Deployment:**
- [ ] Continue monitoring for degradation
- [ ] Keep old version available for rollback
- [ ] Document learnings
- [ ] Plan next iteration

### Common Pitfalls to Avoid

1. **Deploying without A/B test** - Always validate in production
2. **Insufficient sample size** - Wait for statistical power
3. **Ignoring practical significance** - 0.1% improvement may not be worth it
4. **No rollback plan** - Always have escape hatch
5. **Deploying to 100% immediately** - Use gradual rollout
6. **Not monitoring post-deployment** - Models can degrade

### Next Steps

**Session 9.7 Preview: Complete MLOps Project**
- Integrate all concepts (serialization → deployment → monitoring → A/B testing)
- Build end-to-end ML system
- Create portfolio-worthy project
- CI/CD for ML (GitHub Actions)

### Resources

- [A/B Testing for Data Science](https://www.oreilly.com/library/view/a-b-testing-for/9781492043669/)
- [Canary Deployments](https://martinfowler.com/bliki/CanaryRelease.html)
- [Statistical Power Analysis](https://www.statsmodels.org/stable/stats.html#power-and-sample-size-calculations)
- [MLOps Best Practices](https://ml-ops.org/)